# Flower Classification using K-Nearest Neighbors (KNN)
### Iris Flower Species Classification
BSc Artificial Intelligence and Machine Learning — First Year Project

This notebook builds a K-Nearest Neighbors classifier to identify iris flower species
(*setosa*, *versicolor*, *virginica*) from four measurements: sepal length, sepal width,
petal length, and petal width.

**Pipeline:** load data → explore → visualize → scale features → tune K with
cross-validation → evaluate on a held-out test set → predict on new samples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("=" * 60)
print("FLOWER CLASSIFICATION USING K-NEAREST NEIGHBORS (KNN)")
print("Iris Flower Species Classification")
print("=" * 60)

## 1. Loading the Iris Dataset

In [ ]:
iris = load_iris()
X = iris.data   # Features: sepal length, sepal width, petal length, petal width
y = iris.target  # Target: species (0=setosa, 1=versicolor, 2=virginica)

# Create a DataFrame for easier exploration
iris_df = pd.DataFrame(X, columns=iris.feature_names)
iris_df['species'] = y
iris_df['species_name'] = [iris.target_names[i] for i in y]

print("Dataset loaded successfully!")
print(f"Number of samples: {len(iris_df)}")
print(f"Number of features: {X.shape[1]}")
print(f"Target classes: {list(iris.target_names)}")

## 2. Exploring the Dataset

In [ ]:
print("First 5 rows of the dataset:")
print(iris_df.head())

In [ ]:
print("Dataset Information:")
print(iris_df.describe())

In [ ]:
print("Class distribution:")
print(iris_df['species_name'].value_counts())

## 3. Data Visualization

In [ ]:
plt.figure(figsize=(15, 10))

# Plot 1: Scatter plot of sepal length vs sepal width
plt.subplot(2, 2, 1)
for species in [0, 1, 2]:
    plt.scatter(X[y == species, 0], X[y == species, 1],
                label=iris.target_names[species], alpha=0.7)
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.title('Sepal Length vs Sepal Width')
plt.legend()

# Plot 2: Scatter plot of petal length vs petal width
plt.subplot(2, 2, 2)
for species in [0, 1, 2]:
    plt.scatter(X[y == species, 2], X[y == species, 3],
                label=iris.target_names[species], alpha=0.7)
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('Petal Length vs Petal Width')
plt.legend()

# Plot 3: Feature distributions
plt.subplot(2, 2, 3)
iris_df[iris.feature_names].boxplot()
plt.title('Feature Distributions')
plt.xticks(rotation=45)

# Plot 4: Class distribution
plt.subplot(2, 2, 4)
iris_df['species_name'].value_counts().plot(
    kind='bar', color=['skyblue', 'lightcoral', 'lightgreen'])
plt.title('Class Distribution')
plt.xlabel('Species')
plt.ylabel('Count')

plt.tight_layout()
plt.savefig('iris_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Preparing Data for Training

The petal measurements clearly separate the species better than sepal measurements
(see the plots above) — *setosa* is linearly separable, while *versicolor* and
*virginica* overlap slightly.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

### Feature scaling

KNN is a **distance-based** algorithm, so a feature measured on a larger numeric
scale (e.g. petal length in cm) can dominate the distance calculation over a
feature on a smaller scale, even if both are equally informative. Standardizing
puts every feature on the same scale (mean 0, standard deviation 1).

The scaler is **fit only on the training data** and then applied to the test
data, so no information from the test set leaks into training.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Training the K-Nearest Neighbors Model

To choose the best value of **K**, we use 5-fold cross-validation **on the
training set only**. This is important: picking K by checking accuracy on the
test set would tune the model on the same data used for the final evaluation,
which overestimates how well it will generalize.

In [ ]:
k_values = list(range(1, 21))
cv_scores = []

print("Testing different K values (5-fold cross-validation on training data):")
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    cv_scores.append(scores.mean())
    print(f"K = {k:2d}: Mean CV Accuracy = {scores.mean():.4f}")

best_k = k_values[int(np.argmax(cv_scores))]
best_cv_accuracy = max(cv_scores)
print(f"\nBest K value: {best_k} with mean CV accuracy: {best_cv_accuracy:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, cv_scores, marker='o')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best K = {best_k}')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Mean Cross-Validation Accuracy')
plt.title('KNN: Accuracy vs. K Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('k_vs_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Training the Final Model

In [ ]:
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)

y_pred_final = final_knn.predict(X_test_scaled)
final_accuracy = accuracy_score(y_test, y_pred_final)

print(f"Final Model Accuracy on Test Set: {final_accuracy:.4f} ({final_accuracy*100:.2f}%)")

## 7. Model Evaluation

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred_final, target_names=iris.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred_final)
cm_df = pd.DataFrame(cm, index=iris.target_names, columns=iris.target_names)
print("Confusion Matrix:")
print(cm_df)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Confusion Matrix - Iris Flower Classification')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Making Predictions on New Data

New samples must be transformed with the **same scaler fitted on the training
data** — otherwise the distances KNN relies on are meaningless.

In [ ]:
# Sample new flower measurements [sepal length, sepal width, petal length, petal width]
new_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],  # Should be setosa
    [6.0, 2.7, 5.1, 1.6],  # Actually versicolor - a borderline case near the versicolor/virginica boundary
    [5.7, 2.8, 4.1, 1.3],  # Should be versicolor
    [6.3, 3.3, 6.0, 2.5],  # Should be virginica
])

new_flowers_scaled = scaler.transform(new_flowers)

print("Predictions for new flowers:")
for i, flower in enumerate(new_flowers):
    prediction = final_knn.predict(new_flowers_scaled[i:i+1])[0]
    probability = final_knn.predict_proba(new_flowers_scaled[i:i+1])[0]

    print(f"\nFlower {i+1}: {list(flower)}")
    print(f"Predicted species: {iris.target_names[prediction]}")
    print(f"Probabilities: {dict(zip(iris.target_names, np.round(probability, 3)))}")

## 9. Feature Correlation Analysis

A quick, rough look at how strongly each feature correlates with the target.
Note this is a **linear correlation heuristic**, not a true feature-importance
score — KNN has no built-in notion of feature importance.

In [ ]:
feature_corr = []
for i, feature in enumerate(iris.feature_names):
    correlation = np.corrcoef(X[:, i], y)[0, 1]
    feature_corr.append((feature, abs(correlation)))

feature_corr.sort(key=lambda x: x[1], reverse=True)
print("Feature correlation with target (absolute values):")
for feature, corr in feature_corr:
    print(f"{feature}: {corr:.4f}")

In [ ]:
print("=" * 60)
print("PROJECT COMPLETED SUCCESSFULLY!")
print(f"Final Model Accuracy: {final_accuracy*100:.2f}%")
print("The model can successfully classify iris flowers into 3 species:")
print(" - Setosa, Versicolor, and Virginica")
print("based on sepal and petal measurements.")
print("=" * 60)